# 02 — The Series Object: Deep Dive & Interview Essentials
> **Interview Prep & Technical Mastery Guide**
> 
> *A comprehensive, battle-tested reference for Python Data Science, Machine Learning, and Data Engineering Interviews.*

---

## 📌 Executive Summary & Interview Expectations
The `pd.Series` is the atomic one-dimensional building block of Pandas. In interviews, questions about `Series` test whether you understand:
1. **Memory Representation**: How a Series wraps a NumPy 1D array or an ExtensionArray, coupled with an explicit, labeled `Index`.
2. **Automatic Index Alignment**: Why arithmetic on two Series aligns on **labels** rather than positional order.
3. **The Membership Operator Gotcha**: Why `'x' in series` checks the **Index**, not the data values.
4. **Data Type Coercion & Missing Values**: The legacy `float64` promotion problem with `np.nan` vs modern nullable integer types (`Int64`, `pd.NA`).
5. **Modern API Evolution**: `.to_numpy()` vs `.values`, `is_monotonic_increasing` vs deprecated `is_monotonic`, and replacing deprecated `fill_method` in `pct_change()`.

## 1. Environment Setup & Library Versions

In [1]:
import numpy as np
import pandas as pd

print(f"Pandas Version: {pd.__version__}")
print(f"NumPy Version:  {np.__version__}")

Pandas Version: 3.0.6
NumPy Version:  2.5.3


## 2. Creating a Series from Python Objects

A `Series` combines two components:
1. **Values**: A 1-dimensional array of homogeneous data.
2. **Index**: A sequence of axis labels (defaulting to a 0-indexed `RangeIndex`).

> 💡 **Interview Note — Constructor Signature**:
> `pd.Series(data=None, index=None, dtype=None, name=None, copy=False)`
> - If `data` is a **dict**, dictionary keys automatically become the index labels, and values become the series values!
> - If `data` is a **set**, Pandas raises a `TypeError` because sets are unordered collections without deterministic indexing; you must convert to a `list` first.

In [2]:
# 1. Empty Series with explicit dtype (avoids DeprecationWarning)
empty = pd.Series(dtype="object")
empty

Series([], dtype: object)

In [3]:
# 2. Series from Python List
ice_cream_flavors = ["Chocolate", "Vanilla", "Strawberry", "Rum Raisin"]
pd.Series(data=ice_cream_flavors)

0     Chocolate
1       Vanilla
2    Strawberry
3    Rum Raisin
dtype: str

In [4]:
# 3. Series with custom Index labels
days_of_week = ("Monday", "Wednesday", "Friday", "Saturday")
pd.Series(data=ice_cream_flavors, index=days_of_week, name="Flavors")

Monday        Chocolate
Wednesday       Vanilla
Friday       Strawberry
Saturday     Rum Raisin
Name: Flavors, dtype: str

In [5]:
# 4. Series of booleans
bunch_of_bools = [True, False, False]
pd.Series(bunch_of_bools)

0     True
1    False
2    False
dtype: bool

In [6]:
# 5. Stock prices with custom time-of-day index
stock_prices = [985.32, 950.44]
time_of_day = ["Open", "Close"]
pd.Series(data=stock_prices, index=time_of_day, name="GOOG")

Open     985.32
Close    950.44
Name: GOOG, dtype: float64

In [7]:
# 6. Integer Series
lucky_numbers = [4, 8, 15, 16, 23, 42]
pd.Series(lucky_numbers)

0     4
1     8
2    15
3    16
4    23
5    42
dtype: int64

## 3. Missing Values & Dtype Coercion Mechanics

### ⚠️ Top Interview Trap: The `np.nan` Float Coercion Problem
- In NumPy, `np.nan` is a IEEE-754 **floating-point value** (`float`).
- In classic Pandas, introducing `np.nan` into an integer array **silently coerces the entire Series to `float64`**!
- In modern Pandas (1.0+), you can use the nullable integer type (`dtype="Int64"`, uppercase `I`) and `pd.NA` to preserve integer representations without unwanted float coercion.

In [8]:
# Classic NumPy float coercion: integers 94, 88 coerced to float64 due to np.nan
temperatures = [94, 88, np.nan, 91]
s_temp = pd.Series(data=temperatures)
print(f"Dtype with np.nan: {s_temp.dtype}")
s_temp

Dtype with np.nan: float64


0    94.0
1    88.0
2     NaN
3    91.0
dtype: float64

In [9]:
# Modern Nullable Integer Dtype (preserves integers alongside pd.NA)
modern_ints = pd.Series([94, 88, None, 91], dtype="Int64")
print(f"Modern Nullable Dtype: {modern_ints.dtype}")
modern_ints

Modern Nullable Dtype: Int64


0      94
1      88
2    <NA>
3      91
dtype: Int64

## 4. Constructing Series from Other Python Containers

In [10]:
# From Dictionary: keys become index labels, values become data
calorie_info = {"Cereal": 125, "Chocolate Bar": 406, "Ice Cream Sundae": 342}
diet = pd.Series(calorie_info, name="Calories")
diet

Cereal              125
Chocolate Bar       406
Ice Cream Sundae    342
Name: Calories, dtype: int64

In [11]:
# From Tuple of strings
pd.Series(data=("Red", "Green", "Blue"))

0      Red
1    Green
2     Blue
dtype: str

In [12]:
# From List of Tuples (useful for coordinate pairs or RGB)
rgb_colors = [(120, 41, 26), (196, 165, 45)]
pd.Series(data=rgb_colors)

0     (120, 41, 26)
1    (196, 165, 45)
dtype: object

In [13]:
# From Set: MUST convert to list first because sets are unordered
my_set = {"Ricky", "Bobby"}
pd.Series(list(my_set))

0    Ricky
1    Bobby
dtype: str

In [14]:
# From NumPy random array
np.random.seed(42)
random_data = np.random.randint(1, 101, 8)
pd.Series(random_data)

0    52
1    93
2    15
3    72
4    61
5    21
6    83
7    87
dtype: int64

## 5. Core Series Attributes & Inspection

### ⚠️ Top Interview Distinction: `.values` vs `.to_numpy()` vs `.array`
1. **`.values`**: Legacy attribute. Returns a NumPy `ndarray` for standard types, but may return an `ExtensionArray` for newer types. Avoid in production code.
2. **`.to_numpy()`**: The **recommended modern standard method** since Pandas 0.24+. Allows explicit control over `dtype` and `copy` (e.g. `s.to_numpy(dtype='float32', na_value=0.0)`).
3. **`.array`**: Returns the internal `ExtensionArray` backing the Series without forcing NumPy coercion.

In [15]:
# Underlying array values
print("diet.to_numpy():", diet.to_numpy(), "| Type:", type(diet.to_numpy()))
print("diet.values:    ", diet.values, "| Type:", type(diet.values))
print("diet.array:     ", diet.array, "| Type:", type(diet.array))

diet.to_numpy(): [125 406 342] | Type: <class 'numpy.ndarray'>
diet.values:     [125 406 342] | Type: <class 'numpy.ndarray'>
diet.array:      <NumpyExtensionArray>
[125, 406, 342]
Length: 3, dtype: int64 | Type: <class 'pandas.arrays.NumpyExtensionArray'>


In [16]:
# Series Index and Dtype
print("Index: ", diet.index, "| Type:", type(diet.index))
print("Dtype: ", diet.dtype)
print("Shape: ", diet.shape)
print("Size:  ", diet.size)

Index:  Index(['Cereal', 'Chocolate Bar', 'Ice Cream Sundae'], dtype='str') | Type: <class 'pandas.Index'>
Dtype:  int64
Shape:  (3,)
Size:   3


### 💡 Monotonicity & Uniqueness Attributes
> - `.is_unique`: Returns `True` if all values in the Series are unique.
> - `.is_monotonic_increasing` / `.is_monotonic_decreasing`: Checks whether values strictly increase/decrease.
> - *(Note: In modern Pandas, the legacy `is_monotonic` alias is deprecated in favor of `is_monotonic_increasing`)*.

In [17]:
# Check uniqueness
print(f"diet.is_unique: {diet.is_unique}")
print(f"pd.Series([3, 3]).is_unique: {pd.Series([3, 3]).is_unique}")

diet.is_unique: True
pd.Series([3, 3]).is_unique: False


In [18]:
# Check monotonicity
s_asc = pd.Series([1, 3, 6])
s_mixed = pd.Series([1, 6, 3])

print(f"[1, 3, 6] is_monotonic_increasing: {s_asc.is_monotonic_increasing}")
print(f"[1, 6, 3] is_monotonic_increasing: {s_mixed.is_monotonic_increasing}")

[1, 3, 6] is_monotonic_increasing: True
[1, 6, 3] is_monotonic_increasing: False


## 6. Slicing with `head()`, `tail()`, and `sample()`

In [19]:
values = range(0, 500, 5)
nums = pd.Series(data=values)

print("First 3 elements:")
display(nums.head(3))

print("Last 4 elements:")
display(nums.tail(4))

First 3 elements:


0     0
1     5
2    10
dtype: int64

Last 4 elements:


96    480
97    485
98    490
99    495
dtype: int64

## 7. Statistical & Mathematical Operations

### ⚠️ Top Interview Question: `count()` vs `len()` & `skipna` Handling
- `len(s)`: Total number of rows including nulls ($O(1)$).
- `s.count()`: Number of **non-null** values.
- In Pandas statistical reductions (`.sum()`, `.mean()`, `.std()`, etc.), **`skipna=True` is the default**!
  In raw NumPy (`np.sum`), `NaN` propagates and turns the whole sum into `nan` unless `np.nansum` is used.

In [20]:
numbers = pd.Series([1, 2, 3, np.nan, 4, 5])
print(f"len(numbers):    {len(numbers)} (all slots)")
print(f"numbers.count(): {numbers.count()} (non-null only)")

len(numbers):    6 (all slots)
numbers.count(): 5 (non-null only)


In [21]:
# Aggregations with and without skipna
print(f"sum(skipna=True):  {numbers.sum(skipna=True)}")
print(f"sum(skipna=False): {numbers.sum(skipna=False)}")
print(f"product():         {numbers.product()}")

sum(skipna=True):  15.0
sum(skipna=False): nan
product():         120.0


In [22]:
# Cumulative operations
display(numbers.cumsum())

0     1.0
1     3.0
2     6.0
3     NaN
4    10.0
5    15.0
dtype: float64

### 💡 Financial Metric: Percentage Change (`pct_change`)
> **Modern Deprecation Alert**: In older Pandas versions, `numbers.pct_change(fill_method="pad")` was used. In Pandas 2.1+, `fill_method` is **deprecated**. The idiomatic, future-proof approach is chaining explicit filling: `numbers.ffill().pct_change()`.

In [23]:
# Percentage change with forward fill
numbers.ffill().pct_change()

0         NaN
1    1.000000
2    0.500000
3    0.000000
4    0.333333
5    0.250000
dtype: float64

In [24]:
# Central tendency and dispersion
print(f"Mean:   {numbers.mean():.2f}")
print(f"Median: {numbers.median():.2f}")
print(f"Std:    {numbers.std():.2f}")
print(f"Max:    {numbers.max()}")
print(f"Min:    {numbers.min()}")

Mean:   3.00
Median: 3.00
Std:    1.58
Max:    5.0
Min:    1.0


In [25]:
# Non-numeric series: lexicographical min and max
animals = pd.Series(["koala", "aardvark", "zebra"])
print("Alphabetical Min:", animals.min())
print("Alphabetical Max:", animals.max())

Alphabetical Min: aardvark
Alphabetical Max: zebra


In [26]:
# Comprehensive summary statistics with describe()
numbers.describe()

count    5.000000
mean     3.000000
std      1.581139
min      1.000000
25%      2.000000
50%      3.000000
75%      4.000000
max      5.000000
dtype: float64

In [27]:
# Unique values and count of uniques
authors = pd.Series(["Hemingway", "Orwell", "Dostoevsky", "Fitzgerald", "Orwell"])
print("Unique authors (ndarray):", authors.unique())
print("Number of unique authors:", authors.nunique())

Unique authors (ndarray): <StringArray>
['Hemingway', 'Orwell', 'Dostoevsky', 'Fitzgerald']
Length: 4, dtype: str
Number of unique authors: 4


## 8. Arithmetic Operations & Method Equivalents

Pandas provides both operator overloads and explicit method names:
- `+` $\Longleftrightarrow$ `.add()`
- `-` $\Longleftrightarrow$ `.sub()` / `.subtract()`
- `*` $\Longleftrightarrow$ `.mul()` / `.multiply()`
- `/` $\Longleftrightarrow$ `.div()` / `.divide()`
- `//` $\Longleftrightarrow$ `.floordiv()`
- `%` $\Longleftrightarrow$ `.mod()`

> 💡 **Interview Pro-Tip — Why use methods instead of operators?**
> Methods accept a **`fill_value`** parameter!
> If you add two series with `s1 + s2`, any missing index label in either series becomes `NaN`.
> If you use `s1.add(s2, fill_value=0)`, Pandas fills missing values *before* executing the addition, preserving your data!

In [28]:
s1 = pd.Series(data=[5, np.nan, 15], index=["A", "B", "C"])
s1

A     5.0
B     NaN
C    15.0
dtype: float64

In [29]:
# Scalar addition via operator and method
display(s1 + 3)
display(s1.add(3, fill_value=0))  # Replaces NaN at 'B' with 0 before adding 3!

A     8.0
B     NaN
C    18.0
dtype: float64

A     8.0
B     3.0
C    18.0
dtype: float64

In [30]:
# Other arithmetic methods
print("Multiply by 2:\n", s1.mul(2))
print("\nFloor divide by 4:\n", s1.floordiv(4))
print("\nModulo 3:\n", s1.mod(3))

Multiply by 2:
 A    10.0
B     NaN
C    30.0
dtype: float64

Floor divide by 4:
 A    1.0
B    NaN
C    3.0
dtype: float64

Modulo 3:
 A    2.0
B    NaN
C    0.0
dtype: float64


## 9. Broadcasting & The Core Power of Index Alignment

### ⚠️ Top Interview Question: Index-Aligned Arithmetic
When you execute `s1 + s2`, Pandas does **NOT** add elements based on their positional order (index 0 to index 0).
Instead, it performs an **outer join on the index labels** and adds elements with identical labels.
Any label present in only one Series results in `NaN` unless `fill_value` is supplied!

In [31]:
# Example 1: Same indices
s_a = pd.Series([1, 2, 3], index=["A", "B", "C"])
s_b = pd.Series([4, 5, 6], index=["A", "B", "C"])
s_a + s_b

A    5
B    7
C    9
dtype: int64

In [32]:
# Example 2: Shuffled & partially overlapping indices
s1 = pd.Series(data=[5, 10, 15], index=["A", "B", "C"])
s2 = pd.Series(data=[4, 8, 12, 14], index=["B", "C", "D", "E"])

print("--- Default s1 + s2 (Outer join with NaN for non-overlapping labels): ---")
display(s1 + s2)

print("--- s1.add(s2, fill_value=0) (Preserves values using fill_value): ---")
display(s1.add(s2, fill_value=0))

--- Default s1 + s2 (Outer join with NaN for non-overlapping labels): ---


A     NaN
B    14.0
C    23.0
D     NaN
E     NaN
dtype: float64

--- s1.add(s2, fill_value=0) (Preserves values using fill_value): ---


A     5.0
B    14.0
C    23.0
D    12.0
E    14.0
dtype: float64

In [33]:
# Vectorized Element-wise Equality
s_x = pd.Series([3, 6, np.nan, 12])
s_y = pd.Series([2, 6, np.nan, 12])

# Note: np.nan == np.nan evaluates to False in standard IEEE floating point!
print("s_x == s_y:")
display(s_x.eq(s_y))

s_x == s_y:


0    False
1     True
2    False
3     True
dtype: bool

## 10. Interoperability with Python Built-Ins & The Membership Trap

### ⚠️ Top Interview Trap: The `in` Keyword Gotcha
- In Python dictionaries, `'key' in my_dict` checks the **keys**.
- Because a Series is modeled as an ordered map of `index -> value`, **`x in series` checks the INDEX, NOT the values!**
- To check if a value exists in the Series values, you must check `x in series.values` or `(series == x).any()`!

In [34]:
cities = pd.Series(
    data=["San Francisco", "Los Angeles", "Las Vegas", np.nan],
    index=["CA1", "CA2", "NV1", "NA1"]
)
cities

CA1    San Francisco
CA2      Los Angeles
NV1        Las Vegas
NA1              NaN
dtype: str

In [35]:
# Checking Python built-ins
print("len(cities): ", len(cities))
print("type(cities):", type(cities))
print("list(cities):", list(cities))
print("dict(cities):", dict(cities))

len(cities):  4
type(cities): <class 'pandas.Series'>
list(cities): ['San Francisco', 'Los Angeles', 'Las Vegas', nan]
dict(cities): {'CA1': 'San Francisco', 'CA2': 'Los Angeles', 'NV1': 'Las Vegas', 'NA1': nan}


In [36]:
# ⚠️ THE MEMBERSHIP TRAP:
print("'NV1' in cities:        ", "NV1" in cities)          # True: 'NV1' is an INDEX label
print("'Las Vegas' in cities:  ", "Las Vegas" in cities)    # False: 'Las Vegas' is a VALUE, not index!

# Correct way to check values:
print("'Las Vegas' in cities.values:      ", "Las Vegas" in cities.values)
print("(cities == 'Las Vegas').any():      ", (cities == "Las Vegas").any())

'NV1' in cities:         True
'Las Vegas' in cities:   False
'Las Vegas' in cities.values:       True
(cities == 'Las Vegas').any():       True


## 11. Consolidation Practice: Superhero Power Index

In [37]:
superheroes = [
    "Batman",
    "Superman",
    "Spider-Man",
    "Iron Man",
    "Captain America",
    "Wonder Woman"
]
strength_levels = (100, 120, 90, 95, 110, 120)

heroes = pd.Series(data=strength_levels, index=superheroes, name="Strength")
heroes

Batman             100
Superman           120
Spider-Man          90
Iron Man            95
Captain America    110
Wonder Woman       120
Name: Strength, dtype: int64

In [38]:
print("Top 2 Heroes:\n", heroes.head(2))
print("\nDistinct strength levels:", heroes.nunique())
print("Average strength:        ", round(heroes.mean(), 2))
print("Max strength hero:       ", heroes.idxmax(), f"({heroes.max()})")
print("Min strength hero:       ", heroes.idxmin(), f"({heroes.min()})")
print("\nDoubled Strength:\n", heroes * 2)

Top 2 Heroes:
 Batman      100
Superman    120
Name: Strength, dtype: int64

Distinct strength levels: 5
Average strength:         105.83
Max strength hero:        Superman (120)
Min strength hero:        Spider-Man (90)

Doubled Strength:
 Batman             200
Superman           240
Spider-Man         180
Iron Man           190
Captain America    220
Wonder Woman       240
Name: Strength, dtype: int64


---
## 🎯 12. Technical Interview Corner: Tricky Questions & Drills

### Q1: The Nullable Integer Revolution
**Question**: What is the difference between `pd.Series([1, 2, None])` and `pd.Series([1, 2, None], dtype="Int64")`?

**Answer**:
- `pd.Series([1, 2, None])` defaults to NumPy's `float64` engine because NumPy arrays cannot store `None`/`np.nan` in an integer array without boxing. Thus, `1` becomes `1.0`. This can lead to precision loss for 64-bit integers and breaks strict equality checks (`val == 1`).
- `pd.Series([1, 2, None], dtype="Int64")` uses Pandas' `IntegerArray` extension array backed by a separate boolean mask tracking `NA` values. It stores integers as true 64-bit integers while representing missing data as `pd.NA`.

In [39]:
s_legacy = pd.Series([1, 2, None])
s_modern = pd.Series([1, 2, None], dtype="Int64")

print(f"Legacy Series dtype: {s_legacy.dtype} -> values: {list(s_legacy)}")
print(f"Modern Series dtype: {s_modern.dtype} -> values: {list(s_modern)}")

Legacy Series dtype: float64 -> values: [1.0, 2.0, nan]
Modern Series dtype: Int64 -> values: [np.int64(1), np.int64(2), <NA>]


### Q2: Arithmetic on Shuffled Indices
**Question**: Given two Series:
```python
s1 = pd.Series([10, 20], index=['a', 'b'])
s2 = pd.Series([30, 40], index=['b', 'a'])
```
What is `(s1 + s2)['a']` and why?

**Answer**:
`(s1 + s2)['a']` is `50`.
Pandas does **not** perform position-based addition (which would yield `10 + 30 = 40`). It aligns on the index label `'a'`, adding `s1['a']` (10) and `s2['a']` (40) to produce `50`. This automatic label alignment prevents data corruption caused by out-of-order records.

In [40]:
s1 = pd.Series([10, 20], index=["a", "b"])
s2 = pd.Series([30, 40], index=["b", "a"])

print("s1 + s2:")
print(s1 + s2)
print("Result at index 'a':", (s1 + s2)["a"])

s1 + s2:
a    50
b    50
dtype: int64
Result at index 'a': 50


### Q3: Why does modern Pandas prefer `.to_numpy()` over `.values`?
**Question**: What is the difference between `series.values` and `series.to_numpy()`?

**Answer**:
- `.values` was the historical attribute to access underlying data. However, with the introduction of Pandas ExtensionArrays (e.g. `Categorical`, `DatetimeTZ`, `Nullable Integer`, `ArrowString`), `.values` became inconsistent: sometimes it returned a NumPy `ndarray`, and sometimes an `ExtensionArray`.
- `.to_numpy()` is a formal method with parameters:
  - `dtype`: lets you request a specific output NumPy dtype.
  - `copy`: controls whether a zero-copy view is permitted or a copy is forced.
  - `na_value`: specifies how missing values (`pd.NA`) are represented in the resulting NumPy array.

In [41]:
s_cat = pd.Series(["low", "high", "medium"], dtype="category")
print("Using .values:   ", type(s_cat.values), "->", s_cat.values)
print("Using .to_numpy():", type(s_cat.to_numpy()), "->", s_cat.to_numpy())

Using .values:    <class 'pandas.Categorical'> -> ['low', 'high', 'medium']
Categories (3, str): ['high', 'low', 'medium']
Using .to_numpy(): <class 'numpy.ndarray'> -> ['low' 'high' 'medium']


### Q4: Series Slicing: View vs Copy & Copy-on-Write (CoW)
**Question**: When you take a slice of a Series `sub = s[0:3]` and modify `sub.iloc[0] = 999`, what happens to `s`?

**Answer**:
- **In Classic Pandas (1.x without CoW)**: Slicing created a **view** on the same underlying NumPy memory buffer. Mutating `sub` often mutated `s` as well, or raised a `SettingWithCopyWarning`.
- **In Modern Pandas (2.0+ with Copy-on-Write enabled, default in Pandas 3.0)**: Slices share memory **read-only**. The moment `sub` is modified, Pandas triggers an automatic, lazy copy (*Copy-on-Write*), ensuring that mutations to `sub` **NEVER** unexpectedly corrupt `s`!

In [42]:
# Demonstration of slicing behavior
s_base = pd.Series([10, 20, 30, 40, 50])
sub = s_base.iloc[0:2].copy()  # Explicit copy is always best practice!
sub.iloc[0] = 999

print("s_base unchanged:", list(s_base))
print("sub modified:    ", list(sub))

s_base unchanged: [10, 20, 30, 40, 50]
sub modified:     [999, 20]


### Q5: Hands-on Financial Coding Challenge: Maximum Drawdown
**Challenge**: Given a Series of daily stock prices, calculate the **maximum percentage drawdown (MDD)** in pure vectorized Pandas (no loops!).
$$\text{Drawdown}_t = \frac{\text{Price}_t - \text{RunningMax}_t}{\text{RunningMax}_t}$$

In [43]:
# Vectorized Maximum Drawdown Calculation
prices = pd.Series([100.0, 105.0, 102.0, 98.0, 92.0, 97.0, 89.0, 110.0, 108.0])

# 1. Calculate running peak
running_peak = prices.cummax()

# 2. Calculate percentage drawdown at each step
drawdown = (prices - running_peak) / running_peak

# 3. Maximum Drawdown (the lowest negative return from peak)
max_drawdown = drawdown.min()

print("Price Series:      ", list(prices))
print("Running Peak:      ", list(running_peak))
print("Drawdown Series (%):", [f"{d*100:.1f}%" for d in drawdown])
print(f"\n🔥 Maximum Drawdown: {max_drawdown * 100:.2f}% (Trough occurred at index {drawdown.idxmin()})")

Price Series:       [100.0, 105.0, 102.0, 98.0, 92.0, 97.0, 89.0, 110.0, 108.0]
Running Peak:       [100.0, 105.0, 105.0, 105.0, 105.0, 105.0, 105.0, 110.0, 110.0]
Drawdown Series (%): ['0.0%', '0.0%', '-2.9%', '-6.7%', '-12.4%', '-7.6%', '-15.2%', '0.0%', '-1.8%']

🔥 Maximum Drawdown: -15.24% (Trough occurred at index 6)
